# Set up the Fabric policy - new workspaces

Use this notebook to **create the three workspaces** on a Fabric capacity and set up the policy. It checks the
capacity, creates the workspaces (reusing any that already exist), creates a **policy set** in Workspace 1, adds
three **rules** to it, and **turns it on** for the capacity. The steps are explained just before the cell that runs
them.


| Rule | Applies to | Allows |
| --- | --- | --- |
| 1 | Workspaces 2 and 3 | Lakehouse, Data pipeline, Notebook |
| 2 | Workspace 3 | Data agent |
| 3 | Every other workspace on the capacity | Everything |

So, after up to 15 minutes, **Workspace 2** can't create a Data agent, **Workspace 3** can, and every other
workspace on the capacity works as before.

### Before you start

* Import this notebook **and** `policy_setup_functions.ipynb` into the **same workspace**.
* To create workspaces, you need the **Capacity Contributor** or **Capacity Admin** role on the capacity, and your
  Fabric administrator must allow you to create workspaces (the **Create workspaces** tenant setting).
* Your Fabric administrator has turned on the **Policies** tenant setting for you.
* You're the Admin of the workspaces the notebook creates. If a workspace **already exists**, it's reused, and you
  need the **Contributor**, **Member** or **Admin** role on Workspace 1 (the policy set is created in it). For
  Workspaces 2 and 3, the **Viewer** role is enough.
* To turn the policy set on, you need the **Capacity Admin** role. Without it, steps 1-4 still run, and step 5
  prints a message for your Capacity Admin.

### How to run

Type the capacity ID and workspace names in the **Settings** cell, and then select **Run all**. Not sure of the
capacity ID? Run it anyway: if the ID isn't right, step 1 lists the capacities you can use, with their IDs. Each
step says what it did. If something goes wrong, the notebook stops and tells you what to do. It's safe to run
again.

## Settings - type the capacity ID and workspace names here

In [ ]:
CAPACITY_ID = "00000000-0000-0000-0000-000000000000"   # the Fabric capacity to create the workspaces on

WORKSPACE_1 = "Policy-Admin-Store"   # stores the policy set
WORKSPACE_2 = "Policy-DA-Blocked"    # can create only Lakehouse, Data pipeline and Notebook
WORKSPACE_3 = "Policy-DA-Allowed"    # can also create a Data agent

POLICY_SET_NAME = "Fabric_ItemCreation_Governance"   # no spaces

## Load the setup functions

Loads the notebook **policy_setup_functions**, which has the code for every step below. It must be in the **same
workspace** as this notebook. If this cell says the notebook can't be found, import `policy_setup_functions.ipynb` into this
workspace (keep its name), and run again.

In [ ]:
%run policy_setup_functions

## Set up the policy - steps 1 to 5

The cell below runs all five steps, in order. Each step prints what it did. If a step fails, the notebook stops
there and tells you what to do.

| Step | What it does | Function |
| --- | --- | --- |
| 1. Check the capacity | Checks that the capacity exists, is running, and that you can create workspaces on it (**Capacity Contributor** or **Capacity Admin**). If the ID is wrong, it lists the capacities you can use. It also tells you whether you're a Capacity Admin. | `check_capacity` |
| 2. Create the workspaces | Creates the three workspaces on the capacity; you're the Admin of the ones it creates. Workspaces that already exist are reused and **never moved**. Before creating anything, it stops if Workspace 2 or 3 already exists on another capacity, or if Workspace 1 already exists and you're only a **Viewer** there. | `create_workspaces` |
| 3. Create the policy set | Creates the policy set in Workspace 1. It stays **off** until step 5. If it already exists, it's reused. | `create_policy_set` |
| 4. Add the rules | Sets the three rules from the table at the top. Running it again replaces them, so they're never duplicated. | `add_rules` |
| 5. Turn on the policy set | Turns it on for the capacity. Needs the **Capacity Admin** role; without it, the policy set and rules stay saved (but off), and you get a message to send to your Capacity Admin. | `turn_on_policy_set` |

The functions are in `policy_setup_functions`, and the rules are in its **Rules** section.

In [ ]:
capacity = check_capacity(CAPACITY_ID)                                              # Step 1
ws1, ws2, ws3 = create_workspaces(capacity, WORKSPACE_1, WORKSPACE_2, WORKSPACE_3)  # Step 2
policy_set = create_policy_set(ws1, POLICY_SET_NAME)                                # Step 3
add_rules(ws1, policy_set, ws2, ws3)                                                # Step 4
turn_on_policy_set(ws1, policy_set, capacity)                                       # Step 5

## Try it

After up to 15 minutes:

| Where | Create a... | Result |
| --- | --- | --- |
| Workspace 2 | Data agent | Blocked |
| Workspace 3 | Data agent | Allowed |
| Workspace 2 or 3 | Lakehouse, Data pipeline or Notebook | Allowed |
| Workspace 2 or 3 | Anything else, such as a Warehouse | Blocked |
| Workspace 1, or any other workspace | Anything | Allowed |